# 04 · Evaluation metrics: calibration, cost and coverage

The three plots and one table that decide whether a decision model is shippable.

If matplotlib is installed the figures are drawn; otherwise the numbers are printed.


In [ ]:
import numpy as np
import pandas as pd
from slm_decision import (DecisionSpec, CostOptimalPolicy, TemperatureCalibrator,
                          ece, brier, expected_cost, risk_coverage, coverage_at_risk,
                          per_class_recall)

spec = DecisionSpec(
    name="loan_triage",
    actions=["approve", "decline"],
    costs={("approve", "bad"): 120.0, ("approve", "good"): 0.0,
           ("decline", "bad"): 0.0, ("decline", "good"): 24.0},
)

rng = np.random.default_rng(3)
n = 4000
u = rng.normal(0, 1, n)
y = (rng.random(n) < 1 / (1 + np.exp(-u))).astype(int)
p_raw = 1 / (1 + np.exp(-3.0 * u))            # over-confident by construction
print("n =", n, " base rate =", round(y.mean(), 3))

In [ ]:
cal = TemperatureCalibrator().fit(p_raw, y)
p_cal = cal.transform(p_raw)

table = pd.DataFrame([
    {"scores": "raw", "ece": round(ece(p_raw, y), 4), "brier": round(brier(p_raw, y), 4)},
    {"scores": "temperature", "ece": round(ece(p_cal, y), 4), "brier": round(brier(p_cal, y), 4)},
])
print(table.to_string(index=False))
print("fitted temperature: %.3f (scores are ~3x too sharp)" % cal.temperature)

## Reliability diagram

In [ ]:
bins = np.linspace(0, 1, 11)
idx = np.clip(np.digitize(p_cal, bins) - 1, 0, 9)
conf = [p_cal[idx == b].mean() for b in range(10) if (idx == b).any()]
acc = [y[idx == b].mean() for b in range(10) if (idx == b).any()]

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.plot([0, 1], [0, 1], "--", color="#999", label="perfect")
    ax.plot(conf, acc, "o-", label=f"calibrated (ECE {ece(p_cal, y):.3f})")
    ax.set_xlabel("predicted probability"); ax.set_ylabel("observed frequency")
    ax.legend(); plt.tight_layout(); plt.show()
except ImportError:
    print(pd.DataFrame({"predicted": np.round(conf, 3), "observed": np.round(acc, 3)}))

## The cost curve

Where accuracy says 0.5, the cost matrix says something else.


In [ ]:
thresholds = np.linspace(0.01, 0.99, 99)
costs = []
for t in thresholds:
    actions = ["approve" if pi >= t else "decline" for pi in p_cal]
    costs.append(expected_cost(spec, actions, y))
costs = np.array(costs)
t_star = thresholds[costs.argmin()]

print("cost-optimal threshold : %.2f  (%.2f per case)" % (t_star, costs.min()))
print("cost at 0.50           : %.2f per case" % costs[49])
print("saving                 : %.0f%%" % (100 * (1 - costs.min() / costs[49])))

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 3))
    plt.plot(thresholds, costs)
    plt.axvline(t_star, color="#b91c1c", ls="--", label=f"optimal {t_star:.2f}")
    plt.axvline(0.5, color="#666", ls=":", label="naive 0.5")
    plt.xlabel("threshold on P(good)"); plt.ylabel("cost per case")
    plt.legend(); plt.tight_layout(); plt.show()
except ImportError:
    print("threshold %.2f -> %.2f per case" % (t_star, costs.min()))

## Risk-coverage: how much can you automate?

The one plot that turns "how accurate is it" into "how much can we automate safely".


In [ ]:
# correctness is measured against the cost-minimising action for the realised outcome.
# NB: the risk-coverage curve uses the 0.5 "accuracy" rule, not the cost threshold --
# a policy that declines everything above 0.83 is deliberately conservative, and its
# error on the covered slice is not what selective prediction measures.
ideal = np.where(y == 1, "approve", "decline")
preds = np.where(p_cal >= 0.5, "approve", "decline")
cov, risk = risk_coverage(p_cal, ideal, preds, n_points=12)
print(pd.DataFrame({"coverage": cov.round(3), "error_on_covered": risk.round(4)}).to_string(index=False))
print()
for target in (0.02, 0.05, 0.10):
    print("coverage at %.0f%% risk: %.3f" % (100 * target, coverage_at_risk(p_cal, ideal, preds, target)))

## Per-class recall with support

A class that has collapsed hides behind a good average.


In [ ]:
print(pd.DataFrame(per_class_recall(preds, ideal)).T.to_string())

## Reading the output like an operator

| Observation | Interpretation | Action |
|---|---|---|
| Cost-optimal threshold is far from 0.5 | Cost asymmetry dominates | Publish the derivation, wire the threshold in |
| ECE drops after calibration | Model was over-confident, ranking intact | Ship temperature scaling, refit monthly |
| Coverage at 5% risk is ~0.7 | 30% needs review to hold the budget | Check reviewer capacity before launch |
| A class with low recall on small support | Under-learned rare class | Oversample or merge — do not ship as-is |
